In [1]:
# ----------------------------------------
# DOCUMENT EXTRACTION POC
# ----------------------------------------
# This notebook will:
#
# 1. Take uploaded document/image
# 2. Run OCR on the document
# 3. Extract text from document
# 4. Extract known form fields
# 5. Save structured JSON output
#
# ----------------------------------------
# IMPORT REQUIRED LIBRARIES
# ----------------------------------------


# easyocr
# OCR engine used for extracting text from images/scanned documents
import easyocr


# Path
# Used for safer file and folder handling
# Helps check if file exists before processing
from pathlib import Path


# PDF text extraction for digital PDFs
import pdfplumber


# datetime
# Used for generating date and time
# Helpful for unique output filenames
from datetime import datetime


# json
# Used for saving extracted structured data as JSON
import json


# ----------------------------------------
# PROJECT START CONFIRMATION
# ----------------------------------------

print("Document Extraction POC Started Successfully")

Document Extraction POC Started Successfully


In [2]:
# ----------------------------------------
# CREATE REQUIRED PROJECT FOLDERS
# ----------------------------------------
# We will keep uploaded/input files inside:
# uploads/
#
# We will keep generated output files inside:
# outputs/
#
# This prevents messy project structure.


# Folder where user-uploaded documents/images will be placed
uploads_folder = Path("uploads")


# Folder where extracted text and JSON output will be saved
outputs_folder = Path("outputs")


# Create uploads folder if it does not already exist
uploads_folder.mkdir(exist_ok=True)


# Create outputs folder if it does not already exist
outputs_folder.mkdir(exist_ok=True)


# Confirmation
print("Folders are ready")
print("Upload folder:", uploads_folder.resolve())
print("Output folder:", outputs_folder.resolve())

Folders are ready
Upload folder: /Users/anuradhakumari/Library/Mobile Documents/com~apple~CloudDocs/KAINest/document_extractor/uploads
Output folder: /Users/anuradhakumari/Library/Mobile Documents/com~apple~CloudDocs/KAINest/document_extractor/outputs


In [3]:
# ----------------------------------------
# FILE NAME INPUT WIDGET
# ----------------------------------------
# This creates a real input box
# inside Jupyter/VS Code notebook.


import ipywidgets as widgets
from IPython.display import display


# Create text input box
file_input = widgets.Text(
    
    value="",
    
    placeholder="Enter uploaded file name",
    
    description="File:",
    
    disabled=False
)


# Display input box
display(file_input)

Text(value='', description='File:', placeholder='Enter uploaded file name')

In [4]:
# Uploaded file name inside uploads/ folder
#file_name = "AnjaliSharma.png"

# Read value entered in widget
file_name = file_input.value.strip()

# Prevent empty filename
if file_name == "":
    
    raise ValueError(
        "File name cannot be empty"
    )


# Create complete file path
file_path = uploads_folder / file_name


# Check if file exists
if not file_path.exists():
    
    raise FileNotFoundError(
        f"File not found: {file_path}"
    )


# Check if selected path is actually a file
if not file_path.is_file():
    
    raise ValueError(
        f"Selected path is not a valid file: {file_path}"
    )


# Confirmation
print("File found successfully")
print("Selected File:", file_path)

File found successfully
Selected File: uploads/ArjunMehta.png


In [5]:
# ----------------------------------------
# DETECT ACTUAL FILE TYPE
# ----------------------------------------
# Do NOT trust only file extension.
#
# Example:
# fake.pdf may actually be PNG image.
#
# So we check file signature (magic bytes)
# from actual file content.


# Read first few bytes from file
with open(file_path, "rb") as file:
    
    # Read first 10 bytes
    file_header = file.read(10)


# ----------------------------------------
# DETECT FILE TYPE
# ----------------------------------------

detected_file_type = "unknown"


# PDF signature
if file_header.startswith(b"%PDF"):
    
    detected_file_type = "pdf"


# PNG signature
elif file_header.startswith(b"\x89PNG"):
    
    detected_file_type = "png"


# JPG/JPEG signature
elif file_header.startswith(b"\xff\xd8"):
    
    detected_file_type = "jpg"


# DOCX/XLSX/PPTX signature
# Office files are ZIP-based
elif file_header.startswith(b"PK"):
    
    detected_file_type = "office_document"


# ----------------------------------------
# SHOW DETECTED FILE TYPE
# ----------------------------------------

print(
    f"Detected File Type: {detected_file_type}"
)

Detected File Type: png


In [6]:
# ----------------------------------------
# LOAD OCR MODEL
# ----------------------------------------
# OCR = Optical Character Recognition
#
# OCR converts:
# image text → machine-readable text
#
# EasyOCR internally uses deep learning models
# to detect and recognize characters.


# ['en']
# Means:
# Load OCR model trained for English text.
#
# Later for multilingual support:
# ['en', 'hi']
# can be used for English + Hindi.


# gpu=False
# Means:
# Use CPU instead of GPU.
#
# CPU = normal computer processor
# GPU = faster processor mainly used for AI/deep learning
#
# We use CPU for now because:
# - easier setup
# - works on every machine
# - enough for current POC/testing


# reader
# This becomes our OCR engine object.
#
# Later we will use:
# reader.readtext(file_path)

reader = easyocr.Reader(
    ['en'],
    gpu=False
)


# Confirmation message
print("OCR model loaded successfully")

Using CPU. Note: This module is much faster with a GPU.


OCR model loaded successfully


In [7]:
# ----------------------------------------
# RUN OCR ON SELECTED FILE
# ----------------------------------------
# readtext()
# is the main OCR function in EasyOCR.
#
# It:
# 1. Reads the image/document
# 2. Detects text regions
# 3. Recognizes characters
# 4. Returns OCR results
#
#
# OCR output format:
#
# [
#   (
#       coordinates,
#       extracted_text,
#       confidence_score
#   ),
#   ...
# ]
#
#
# Example:
#
# (
#   [[247,71],[883,71],[883,111],[247,111]],
#   'EMPLOYEE ENROLLMENT FORM',
#   0.9794
# )
#
#
# PART 1 → coordinates
# Where text exists in image
#
# PART 2 → extracted_text
# Actual detected text
#
# PART 3 → confidence_score
# OCR confidence between 0 and 1


# Run OCR
results = reader.readtext(
    str(file_path)
)


# Confirmation message
print("OCR completed successfully")


# Show total detected text regions
print(
    f"Total Text Regions Detected: {len(results)}"
)

/Users/anuradhakumari/Library/Mobile Documents/com~apple~CloudDocs/KAINest/document_extractor/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


OCR completed successfully
Total Text Regions Detected: 22


In [8]:
# ----------------------------------------
# CREATE CLEAN OCR TEXT
# ----------------------------------------
# OCR results currently contain:
# - coordinates
# - text
# - confidence
#
# For later processing:
# - field extraction
# - regex matching
# - JSON generation
#
# We mainly need clean text only.
#
# This step converts OCR output into:
# one readable text block.


# Empty list to store clean text lines
clean_text = []


# Loop through OCR results
for item in results:
    
    
    # item[1]
    # contains extracted OCR text
    text = item[1]
    
    
    # Add text into clean_text list
    clean_text.append(text)


# ----------------------------------------
# COMBINE ALL TEXT LINES
# ----------------------------------------
# "\n"
# means new line
#
# Example:
#
# line1
# line2
# line3

final_text = "\n".join(clean_text)


# ----------------------------------------
# SHOW FINAL CLEAN TEXT
# ----------------------------------------

print(final_text)

EMPLOYEE ENROLLMENT FORM
Employee Name:
Arjun Mehta
Employee ID:
EMP-9153
Date of Birth:
28/08/1992
Date of Joining:
12/06/2026
Department:
Product Management
Designation:
Associate Product Manager
Mobile Number:
9832156487
Email:
arjun.mehta@example.com
Address:
8B, Skyline Towers, Bannerghatta Road, Bengaluru; Karnataka
Pincode:
560076
Signature:


In [9]:
# ----------------------------------------
# FORM SCHEMA / KNOWN FORM KEYS
# ----------------------------------------
# Supervisor/client provides:
# - form structure
# - field names to extract
#
# Our system uses these known keys
# to extract values from uploaded forms.
#
# Left Side:
# Final JSON field name
#
# Right Side:
# Actual label expected inside form


form_schema = {
    
    "employee_name": "Employee Name:",
    
    "employee_id": "Employee ID:",
    
    "date_of_birth": "Date of Birth:",
    
    "date_of_joining": "Date of Joining:",
    
    "department": "Department:",
    
    "designation": "Designation:",
    
    "mobile_number": "Mobile Number:",
    
    "email": "Email:",
    
    "address": "Address:",
    
    "pincode": "Pincode:"
}


# ----------------------------------------
# CONFIRM SCHEMA CREATED
# ----------------------------------------

print("Form schema created successfully")


# Show all fields system will extract
for key in form_schema:
    
    print(key)

Form schema created successfully
employee_name
employee_id
date_of_birth
date_of_joining
department
designation
mobile_number
email
address
pincode


In [10]:
# ----------------------------------------
# EXTRACT VALUES FROM OCR TEXT
# ----------------------------------------
# Goal:
# Convert OCR text into structured JSON data.
#
# Example:
#
# Employee Name:
# Ramesh Kumar
#
# becomes:
#
# {
#   "employee_name": "Ramesh Kumar"
# }
#
#
# Logic Used:
#
# If current OCR line matches a known form label,
# then:
# next line is treated as the field value.


# Split clean OCR text into separate lines
lines = final_text.split("\n")


# Empty dictionary to store extracted values
extracted_json = {}


# Loop through all OCR text lines
for index, line in enumerate(lines):
    
    
    # Remove extra spaces
    current_line = line.strip()
    
    
    # Check every known form field
    for json_key, form_label in form_schema.items():
        
        
        # If OCR line matches known form label
        if current_line == form_label:
            
            
            # Check if next line exists
            if index + 1 < len(lines):
                
                
                # Next line becomes extracted value
                extracted_json[json_key] = (
                    lines[index + 1].strip()
                )
            
            
            # If value missing
            else:
                
                extracted_json[json_key] = ""


# ----------------------------------------
# SHOW EXTRACTED JSON DATA
# ----------------------------------------

print(extracted_json)

{'employee_name': 'Arjun Mehta', 'employee_id': 'EMP-9153', 'date_of_birth': '28/08/1992', 'date_of_joining': '12/06/2026', 'department': 'Product Management', 'designation': 'Associate Product Manager', 'mobile_number': '9832156487', 'email': 'arjun.mehta@example.com', 'address': '8B, Skyline Towers, Bannerghatta Road, Bengaluru; Karnataka', 'pincode': '560076'}


In [11]:
# ----------------------------------------
# SAVE EXTRACTED JSON OUTPUT
# ----------------------------------------
# Final extracted structured data
# will now be saved as:
# .json file
#
# JSON is useful because:
# - structured format
# - easy API response
# - easy database insertion
# - frontend friendly
# - industry standard format


# Get uploaded file name without extension
#
# Example:
# sample_employee_form.png
#
# becomes:
# sample_employee_form

base_name = file_path.stem


# ----------------------------------------
# GENERATE CURRENT TIMESTAMP
# ----------------------------------------
# Used for unique JSON filenames

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


# ----------------------------------------
# CREATE JSON OUTPUT FILE NAME
# ----------------------------------------
#
# Example:
# sample_employee_form_20260525_190530.json

json_output_file = (
    f"{base_name}_{timestamp}.json"
)


# Create complete output file path
json_output_path = (
    outputs_folder / json_output_file
)


# ----------------------------------------
# SAVE JSON FILE
# ----------------------------------------
# indent=4
# makes JSON more readable
#
# ensure_ascii=False
# supports Hindi/regional language text

with open(
    json_output_path,
    "w",
    encoding="utf-8"
) as file:
    
    
    json.dump(
        extracted_json,
        file,
        indent=4,
        ensure_ascii=False
    )


# ----------------------------------------
# CONFIRM JSON SAVED
# ----------------------------------------

print("JSON output saved successfully")
print("Saved JSON File:", json_output_path)

JSON output saved successfully
Saved JSON File: outputs/ArjunMehta_20260525_200621.json
